# Milvus 3.0 端到端实战

对应文档 [7.5 端到端实战](./7.5-Milvus3.0端到端实战)。一条链跑通：**湖上 Parquet → 外部 Collection → 索引刷新 → 检索 → 快照恢复 → 增量刷新 → 清理**。

前置条件（环境搭建见 [7.1](./7.1-Milvus3.0环境升级)）：

- Docker Desktop 在跑，Milvus **v3.0.2** standalone 已启动；MinIO 端口 `9000`、Milvus 端口 `19530` 映射到本机；
- 从上往下顺序执行，所有输出都是本地实跑结果。

## 1. 安装依赖

教程基线 `pymilvus 3.0.2`，与服务端版本对应。

In [1]:
%pip install -q pymilvus==3.0.2 pyarrow boto3 numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: C:\Users\liuxiao\GenericAgent\temp\milvus3_lab\venv\Scripts\python.exe -m pip install --upgrade pip


## 2. 环境检查

两个服务都必须可达；连不上先回去查 7.1 的 compose。

In [2]:
import boto3
from botocore.config import Config
from pymilvus import MilvusClient

client = MilvusClient(uri="http://127.0.0.1:19530")
print("Milvus server:", client.get_server_version())

s3 = boto3.client(
    "s3",
    endpoint_url="http://localhost:9000",
    aws_access_key_id="minioadmin",
    aws_secret_access_key="minioadmin",
    region_name="us-east-1",
    config=Config(signature_version="s3v4", s3={"addressing_style": "path"}),
)
print("MinIO buckets:", [b["Name"] for b in s3.list_buckets()["Buckets"]])

Milvus server: 3.0.2


MinIO buckets: ['a-bucket', 'lake-demo']


## 3. 准备数据：生成 Parquet，传上 MinIO（对应 7.5.1）

1000 行商品数据，四列：`id` / `product_name` / `price` / `embedding`（768 维 float32）。真实项目里这一步换成你自己的湖上数据。

In [3]:
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq

rng = np.random.default_rng(42)
N, DIM = 1000, 768
ids = np.arange(1, N + 1)
names = [f"product_{i:04d}" for i in ids]
price = np.round(rng.uniform(10, 1000, size=N), 2)
emb = rng.standard_normal((N, DIM)).astype(np.float32)

tbl = pa.table({
    "id": pa.array(ids),
    "product_name": pa.array(names),
    "price": pa.array(price),
    "embedding": pa.array(emb.tolist(), pa.list_(pa.float32(), DIM)),
})
pq.write_table(tbl, "products.parquet", compression="snappy")
print("rows:", tbl.num_rows)

s3.create_bucket(Bucket="lake-demo") if "lake-demo" not in [b["Name"] for b in s3.list_buckets().get("Buckets", [])] else None
s3.upload_file("products.parquet", "lake-demo", "products75/products.parquet")
print([(o["Key"], o["Size"]) for o in s3.list_objects_v2(Bucket="lake-demo", Prefix="products75/")["Contents"]])

rows: 1000
[('products75/products.parquet', 3714148)]


## 4. 建外部 Collection（对应 7.5.2 第一步）

两个实测坑：

- **不要声明 `is_primary=True`**——外部集合禁止用户定义主键，主键由 Milvus 自动生成（`auto_id: True`）；
- **`endpoint_url` 必须写 Milvus 容器视角的地址**（compose 网络里是 `http://minio:9000`）；写 `localhost` 会指向 Milvus 容器自己，必失败。

In [4]:
import json
from pymilvus import MilvusClient, DataType

EXT_SPEC = json.dumps({
    "format": "parquet",
    "extfs": {
        "access_key_id": "minioadmin",
        "access_key_value": "minioadmin",
        "region": "us-east-1",
        "cloud_provider": "aws",
        "endpoint_url": "http://minio:9000",   # Milvus 容器视角，不是 localhost
        "use_ssl": "false",
        "use_virtual_host": "false",
    },
})

schema = MilvusClient.create_schema(
    external_source="s3://lake-demo/products75/",
    external_spec=EXT_SPEC,
)
schema.add_field(field_name="id", datatype=DataType.INT64, external_field="id")
schema.add_field(field_name="product_name", datatype=DataType.VARCHAR, max_length=512, external_field="product_name")
schema.add_field(field_name="price", datatype=DataType.DOUBLE, external_field="price")
schema.add_field(field_name="embedding", datatype=DataType.FLOAT_VECTOR, dim=768, external_field="embedding")

if client.has_collection(collection_name="catalog"):
    client.drop_collection(collection_name="catalog")
client.create_collection(collection_name="catalog", schema=schema)
print("auto_id =", client.describe_collection("catalog")["auto_id"])   # True：主键由 Milvus 生成

auto_id = True


## 5. 建索引并刷新（对应 7.5.2 第二步）

`refresh` 让 Milvus 扫描湖上文件生成数据清单——不搬数据、不重建索引。

实测坑：**`refresh` 返回 `job_id`，任务是异步执行的**。必须轮询到 `RefreshCompleted` 再 `load`，否则 load 到的是刷新前的版本，搜索直接空结果。

In [5]:
import time

index_params = client.prepare_index_params()
index_params.add_index(field_name="id", index_type="AUTOINDEX")
index_params.add_index(field_name="product_name", index_type="AUTOINDEX")
index_params.add_index(field_name="embedding", index_type="AUTOINDEX", metric_type="COSINE")
client.create_index(collection_name="catalog", index_params=index_params)

def refresh_and_wait(coll):
    t0 = time.time()
    job_id = client.refresh_external_collection(collection_name=coll)
    while True:
        jobs = [j for j in client.list_refresh_external_collection_jobs() if str(j.job_id) == str(job_id)]
        st = str(jobs[0].state)
        if st == "RefreshCompleted":
            break
        if st == "RefreshFailed":
            raise RuntimeError(jobs[0].reason)
        time.sleep(1)
    print(f"refresh 耗时: {time.time()-t0:.2f}s")

refresh_and_wait("catalog")
client.load_collection(collection_name="catalog")

refresh 耗时: 1.03s


## 6. 检索：向量搜索 + 价格过滤（对应 7.5.2 第三步）

拿第 1 行的向量当查询向量；`price < 500` 下推到服务端，只传回命中行。

In [6]:
query_vec = pq.read_table("products.parquet").column("embedding")[0].as_py()

res = client.search(
    collection_name="catalog",
    data=[query_vec],
    anns_field="embedding",
    search_params={"metric_type": "COSINE"},
    filter="price < 500",
    limit=5,
    output_fields=["product_name", "price"],
)
for hit in res[0]:
    print(f"{hit['distance']:.4f} | {hit['entity']['product_name']} | {hit['entity']['price']}")

0.0977 | product_0634 | 378.98
0.0952 | product_0398 | 302.41
0.0853 | product_0718 | 110.88
0.0849 | product_0548 | 89.25
0.0806 | product_0218 | 132.52


## 7. 快照备份与恢复（对应 7.5.2 第四步）

四个实测坑：

- 外部集合**不支持 `flush`**，拍快照前不要调（`flush operation is not supported for external collection`）；
- `restore_snapshot` 签名是 `(snapshot_name, source_collection_name, target_collection_name)`，返回 `job_id`；
- `describe_snapshot` 必须同时给 `snapshot_name` 和 `collection_name`；
- **恢复出的集合要先 `load_collection` 才能查**，否则 `collection not loaded`。

In [7]:
if "catalog_base" not in client.list_snapshots(collection_name="catalog"):
    client.create_snapshot(collection_name="catalog", snapshot_name="catalog_base")
print(client.list_snapshots(collection_name="catalog"))

if client.has_collection(collection_name="catalog_restored"):
    client.drop_collection(collection_name="catalog_restored")
job_id = client.restore_snapshot("catalog_base", "catalog", "catalog_restored")
print("job_id =", job_id)
while True:
    st = client.get_restore_snapshot_state(job_id=job_id)
    print(f"State: {st.state}, Progress: {st.progress} %")
    if st.state == "RestoreSnapshotFailed":
        raise RuntimeError(st.reason)
    if st.progress >= 100:
        break
    time.sleep(2)

client.load_collection(collection_name="catalog_restored")   # 恢复出的集合要先 load
print(client.query(collection_name="catalog_restored", filter="price < 500", output_fields=["count(*)"]))

['catalog_base']
job_id = 470246985439621377


State: RestoreSnapshotPending, Progress: 0 %


State: RestoreSnapshotExecuting, Progress: 0 %


State: RestoreSnapshotExecuting, Progress: 0 %


State: RestoreSnapshotCompleted, Progress: 100 %


data: ["{'count(*)': 500}"], extra_info: {}


## 8. 增量：湖上数据变了，再刷一次（对应 7.5.3）

外部数据只读，湖上是唯一的数据主副本：改湖上文件（同名覆盖）→ 重新 `refresh` → Milvus 看到新数据。两个实测坑：刷新不会自动发生；**刷新完成后要 `release` 再 `load`——已加载的集合直接重新 `load` 是幂等的，查到的还是旧数据**。

In [8]:
# 把 id <= 5 的价格改成 1.23，同名覆盖上传，再刷新
t = pq.read_table("products.parquet")
d = t.to_pydict()
d["price"] = [1.23 if pid <= 5 else p for pid, p in zip(d["id"], d["price"])]
tbl_v2 = pa.Table.from_pydict(d, schema=t.schema)
pq.write_table(tbl_v2, "products_v2.parquet", compression="snappy")
s3.upload_file("products_v2.parquet", "lake-demo", "products75/products.parquet")

refresh_and_wait("catalog")
client.release_collection(collection_name="catalog")   # 直接重新 load 是幂等的，必须先 release
client.load_collection(collection_name="catalog")

for r in client.query(collection_name="catalog", filter="id <= 5", output_fields=["id", "price"]):
    print(r["id"], r["price"])

refresh 耗时: 1.02s


1 1.23
2 1.23
3 1.23
4 1.23
5 1.23


## 9. 清理（可选）

删掉演练产生的集合、快照和湖上文件；想保留结果观察就跳过。

In [9]:
client.drop_snapshot(snapshot_name="catalog_base", collection_name="catalog")   # 先删快照：它引用源集合
client.drop_collection(collection_name="catalog_restored")
client.drop_collection(collection_name="catalog")
s3.delete_object(Bucket="lake-demo", Key="products75/products.parquet")
print("cleaned. collections:", client.list_collections())

cleaned. collections: ['demo_fts', 'demo_emblist', 'shop_items', 'products_snapdemo_r', 'nulls_ext', 'demo_emblist_auto', 'demo_emblist_263', 'products_restored', 'products_demo', 'headphones', 'demo_faiss', 'products_dup', 'products_verify', 'fc_demo', 'catalog_r2', 'products_managed', 'products_ext', 'products_snapdemo', 'demo_products', 'demo_noname']


跑完这条链，三件事值得看：

- **存储**：对象存储里只有索引和清单，没有第二份全量副本；
- **刷新延迟**：本地毫秒级（官方口径 150-250 ms，随元数据规模），只扫元数据、不重建索引；
- **恢复**：快照恢复产出新集合（分段复制，官方口径 10-100 倍加速），改业务代码指向新集合即完成回滚。

参考文档：[创建外部 Collection](https://milvus.io/docs/zh/create-an-external-collection.md) · [管理快照](https://milvus.io/docs/zh/manage-snapshots.md) · [快照用例](https://milvus.io/docs/zh/snapshot-use-cases.md)